In [ ]:
from os import listdir as ls
import pandas as pd
import seaborn as sns
from matplotlib.pyplot import subplots
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats

from emu_renewal.inputs import get_gdps, get_country_pop
from emu_renewal.constants import OUTPUTS_PATH, ANALYSIS_NAMES
from emu_renewal.outputs import get_idatas_for_mob_type
from emu_renewal.utils import get_country_short_name
from emu_renewal.outputs import get_param_vals_by_analysis, get_ratios_from_disps, get_median_ratios

set_matplotlib_formats("svg")

https://unstats.un.org/unsd/methodology/m49/overview/#

In [ ]:
job_path = OUTPUTS_PATH / "46693102"
all_countries = ls(job_path)
mob_source = "fb_singletile_mob"

In [ ]:
# Get the dispersion ratios
disp_posts = {c: get_param_vals_by_analysis("dispersion_proc", job_path / c) for c in all_countries}
ratio_dists = get_ratios_from_disps(disp_posts)
ratio_vals = get_median_ratios(ratio_dists, mob_source)

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=[12, 9])
flat_axes = axs.ravel()
analyses = {k: v for k, v in ANALYSIS_NAMES.items() if "no_mob" not in k}
for m, (mob_source, mob_name) in enumerate(analyses.items()):
    ax = flat_axes[m]
    ax.set_title(mob_name)
    idatas, _ = get_idatas_for_mob_type(job_path, all_countries, mob_source)
    mob_exp_df = pd.DataFrame(columns=idatas)
    for iso3 in idatas:
        mob_exp_df[iso3] = idatas[iso3].posterior["mob_exp"].to_dataframe()
    comb_df = pd.DataFrame()
    comb_df["mob_exp"] = mob_exp_df.median()
    comb_df["ratio"] = pd.Series(ratio_vals)
    comb_df["GDP"] = get_gdps(2020)
    comb_df["pop"] = {c: get_country_pop(c) / 1e6 for c in all_countries}
    sns.scatterplot(x="ratio", y="mob_exp", hue="GDP", size="pop", data=comb_df, sizes=(25, 500), ax=ax)
    ax.set_xlabel("dispersion ratio")
    ax.set_ylabel("mobility exponent")
    handles, labels = ax.get_legend_handles_labels()
    ax.get_legend().remove()
ax = flat_axes[-1]
ax.legend(handles=handles, labels=labels, loc="center", ncol=2)
ax.axis("off")

fig.tight_layout()

In [ ]:
fig